# Model shortlist: open-ended IVP accuracy with working allowed (Kaggle)

Every candidate model solves the IVP-OPEN problems (no options), shows its working and ends with
`Answer: y = ...`; SymPy checks the answer. The table at the end ranks the models. Shortlist from
it, then run the full test split for the shortlisted models (last settings cell).

**Before running** (right-hand panel):
- *Session options → Accelerator*: **GPU T4 x2** (one GPU is used) or GPU P100
- *Session options → Internet*: **on**
- *Add-ons → Secrets*: add `HF_TOKEN` (your Hugging Face read token) and tick it for this notebook.
  Only needed for Llama-3.2; the other models are public.
- *Input*: add the dataset made from `kaggle_open_eval_files.zip`

Results: `/kaggle/working/Thesis/results/open_eval/`, zipped at the end as
`/kaggle/working/open_eval_results.zip` (*Output* panel). Generations are saved batch by batch, so
an interrupted run keeps what it finished.

Time (T4, dev = 72 problems): about 3-6 min per model plus the download; about 45 min for all.

In [ ]:
import os, sys, glob, json, math, shutil, subprocess
INPUT = os.environ.get('KAGGLE_INPUT_DIR', '/kaggle/input')        # read-only dataset mount
WORK = os.environ.get('KAGGLE_WORK_DIR', '/kaggle/working/Thesis')  # writable, kept as output
found = glob.glob(os.path.join(INPUT, '**', 'eval_open_ivp.py'), recursive=True)
assert found, f'project files not found under {INPUT}: add the dataset as an Input of this notebook'
SRC = os.path.dirname(found[0])
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
for f in ['eval_open_ivp.py', 'baseline_letter_logprobs.py', 'ivp_open/grade_open.py',
          'ivp_open/ivp_open_dev.jsonl', 'ivp_open/ivp_open_test.jsonl', 'ivp_mcq/grade_mcq.py']:
    assert os.path.exists(f), f'missing {f} in the uploaded dataset (keep the folder structure of the zip)'
print('project files copied from', SRC, 'to', WORK)
ON_KAGGLE = INPUT == '/kaggle/input'
if ON_KAGGLE:
    r = subprocess.run('curl -s -o /dev/null -w "%{http_code}" https://huggingface.co', shell=True,
                       capture_output=True, text=True)
    assert r.stdout.strip() in ('200', '301', '302'), 'no internet: turn Internet on in Session options'
    subprocess.run('pip -q install "transformers>=4.51" accelerate', shell=True)
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN set')
    except Exception as e:
        print('no HF_TOKEN secret (fine unless you run Llama):', type(e).__name__)
import torch, transformers
if ON_KAGGLE:
    assert torch.cuda.is_available(), 'no GPU: set Accelerator to GPU T4 x2 or P100 in Session options'
print('cwd', os.getcwd(), '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Settings: which models, which split

In [ ]:
SPLIT = 'dev'                 # 'dev' (72) for the shortlist; 'test' (792) for the shortlisted models
RUNS = [                      # (model key, extra arguments)
    ('qwen2.5-0.5b-instruct', ''),
    ('qwen2.5-1.5b-instruct', ''),
    ('qwen2.5-3b-instruct', ''),
    ('qwen2.5-math-1.5b-instruct', ''),
    ('qwen3-1.7b', ''),                       # non-thinking; add a run with '--thinking --max-new-tokens 2048' to try thinking
    ('qwen3-4b-instruct-2507', ''),
    ('phi-4-mini-instruct', ''),
    ('llama-3.2-3b-instruct', ''),            # needs the HF_TOKEN secret
]
MAX_NEW_TOKENS = 768
BATCH_SIZE = 8
LIMIT = None                  # leave None for real runs
DELETE_CACHE = True           # delete each model's download after its run (saves disk)
OUT = 'results/open_eval'

## Run every model

In [ ]:
def hf_cache_dir():
    return os.environ.get('HF_HUB_CACHE') or os.path.join(
        os.environ.get('HF_HOME', os.path.expanduser('~/.cache/huggingface')), 'hub')

sys.path.insert(0, WORK)
from baseline_letter_logprobs import MODELS
dirs, failed = [], []
for key, extra in RUNS:
    cmd = (f'{sys.executable} eval_open_ivp.py --model {key} --split {SPLIT} --max-new-tokens {MAX_NEW_TOKENS} '
           f'--batch-size {BATCH_SIZE} --outdir {OUT} {extra}' + (f' --limit {LIMIT}' if LIMIT else ''))
    if '--max-new-tokens' in extra:
        cmd = cmd.replace(f'--max-new-tokens {MAX_NEW_TOKENS} ', '')
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if 'acc=' in l]
    if r.returncode or not line:
        failed.append(key)
        print('   FAILED:\n   ' + '\n   '.join(r.stderr.strip().splitlines()[-6:]), flush=True)
    else:
        print('  ', line[-1].split('  -> ')[0], flush=True)
        dirs.append(line[-1].split('  -> ')[1].strip())
    if DELETE_CACHE and ON_KAGGLE:
        shutil.rmtree(os.path.join(hf_cache_dir(), 'models--' + MODELS[key][0].replace('/', '--')), ignore_errors=True)
print('\nfailed:', failed or 'none')

## Ranking

In [ ]:
subprocess.run(f'{sys.executable} eval_open_ivp.py --table ' + ' '.join(f'"{d}/summary.json"' for d in dirs), shell=True)

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (c - h), 100 * (c + h)

print('\nranking with 95% confidence intervals (dev has only 72 problems: overlapping intervals are ties)')
rows = []
for d in dirs:
    s = json.load(open(f'{d}/summary.json'))
    n = s['meta']['n_items']; k = round(s['accuracy'] * n)
    rows.append((s['accuracy'], s['meta']['model_key'] + (' +thinking' if s['meta']['thinking'] else ''), k, n, s['truncated']))
for acc, name, k, n, tr in sorted(rows, reverse=True):
    lo, hi = wilson(k, n)
    print(f'  {name:34s} {100 * acc:5.1f}%  [{lo:4.1f}, {hi:4.1f}]  ({k}/{n}, truncated {tr})')

## Download the results

In [ ]:
zip_base = os.path.join(os.path.dirname(WORK), 'open_eval_results')
path = shutil.make_archive(zip_base, 'zip', root_dir=WORK, base_dir=OUT)
print('wrote', path, f'({os.path.getsize(path) / 1e6:.2f} MB) - download it from the Output panel on the right')